# 04. Storage Backends

## Learning Objectives
- Understand how backends implement the agent's filesystem
- Learn the characteristics and use cases of the five built-in backends
- Configure path-based routing with `CompositeBackend`
- Implement a custom backend with `BackendProtocol`


In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("Environment setup complete")

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


---
## 1. What is a backend?

Deep Agents' built-in file tools (`ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep`) all operate through a **backend**. `read_file` can return multimodal content for image files.

A backend abstracts the **storage layer** the agent uses to read and write files.

![Backend abstraction](../assets/images/backend_abstraction.png)

### Available backends

| Backend | Storage | Persistence | Use case |
|--------|----------|--------|-------------|
| `StateBackend` | Agent state (LangGraph state) | Within a thread | Temporary scratchpad (default) |
| `FilesystemBackend` | Local disk | Persistent | Local file access, coding agents |
| `LocalShellBackend` | Disk + shell execution | Persistent | Development environments (security caution) |
| `StoreBackend` | LangGraph `BaseStore` | Cross-thread | Long-term memory, user preferences |
| `ContextHubBackend` | LangSmith Hub storage | Persistent | Shared file trees, lazy fetch + write-through |
| `CompositeBackend` | Path-based routing | Mixed | Memory + temporary files |
| **Sandbox** | Modal / Daytona / Deno / local VFS | Isolated | Isolated shell and file execution |

> Current examples pass a prebuilt **`BackendProtocol` instance**. `StoreBackend` should declare a `namespace`, while the actual store is passed through `create_deep_agent(store=...)`. A runtime-aware namespace factory such as `namespace=lambda rt: (...)` remains valid.

In [ ]:
# Verify backend imports
from deepagents.backends import (
    StateBackend,
    FilesystemBackend,
    StoreBackend,
    CompositeBackend,
)
from deepagents.backends.protocol import BackendProtocol

print("All backend classes imported successfully!")


---
## 2. `StateBackend` (Default)

`StateBackend` stores files in agent state (LangGraph state).  
It is **ephemeral**: files live only inside the current conversation thread.

### Characteristics
- Used automatically when you do not pass `backend` to `create_deep_agent()`
- Files survive across agent turns through checkpoints
- Files disappear when the thread ends
- No external storage required


In [ ]:
from deepagents import create_deep_agent

# StateBackend is the default, so you do not need to configure it explicitly
agent = create_deep_agent(
    model=model,
    system_prompt="You are a file-management assistant. Respond in English.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Create a file named 'hello.txt' with the text 'Hello!'. Then check the file list."}]},
    config=lf_config,
)

print(result["messages"][-1].content)


---
## 3. FilesystemBackend — Local Disk Access

This backend lets the agent access the **local filesystem**.

### Key options
- `root_dir` — accessible root directory; defaults to the current directory
- `virtual_mode=True` — the default; blocks `..`, `~`, and absolute paths outside `root_dir`
- `max_file_size_mb` — maximum file size that can be read

`write_file` can replace an existing file at the same path. If the backend supports `delete`, `delete` is also exposed, so write permissions should be reviewed together with deletion risk.

### ⚠️ Security note
> `FilesystemBackend` gives the agent real filesystem access.
> It may read accessible `.env` files, API keys, or credentials.
> In production, combine `virtual_mode=True` with approval policies, or prefer a sandbox backend.

In [ ]:
import tempfile
from pathlib import Path

# Use a temporary directory for safe testing
with tempfile.TemporaryDirectory() as tmp_dir:
    Path(tmp_dir, "sample.txt").write_text("This is a sample file.", encoding="utf-8")
    Path(tmp_dir, "data.csv").write_text("name,age\nAlice,30\nBob,25", encoding="utf-8")

    fs_agent = create_deep_agent(
        model=model,
        system_prompt="You are a file analysis assistant. Respond in English.",
        backend=FilesystemBackend(
            root_dir=tmp_dir,
            virtual_mode=True,
        ),
    )

    result = fs_agent.invoke(
        {"messages": [{"role": "user", "content": "Show me the files in the current directory, read each file, and summarize the contents."}]},
        config=lf_config,
    )

    print(result["messages"][-1].content)


---
## 4. StoreBackend — Cross-Thread Persistent Storage

`StoreBackend` uses a LangGraph `BaseStore` to persist files **across conversation threads**.

### Characteristics
- Different threads can access the same stored files when they share the namespace
- Redis, PostgreSQL, and cloud store implementations can be used behind `BaseStore`
- `namespace` is the isolation boundary. Use a fixed tuple for demos, or `namespace=lambda rt: (...)` to read runtime context.
- Pass a `StoreBackend(namespace=...)` instance together with `create_deep_agent(store=...)`.

### Namespace patterns

```python
# fixed demo namespace
StoreBackend(namespace=lambda rt: ("demo-user",))

# user-scoped runtime context
StoreBackend(namespace=lambda rt: (rt.context.user_id,))
```

Do not confuse the removed backend factory pattern with a namespace factory. A namespace callable remains valid.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

store, checkpointer = InMemoryStore(), MemorySaver()
store_backend = StoreBackend(namespace=lambda rt: ("demo-user",))
store_agent = create_deep_agent(
    model=model, backend=store_backend, store=store,
    checkpointer=checkpointer, system_prompt="You manage notes.",
)
print("StoreBackend agent created")

In [ ]:
# Save a note in thread 1
config_thread1 = {"configurable": {"thread_id": "thread-1"}}

result1 = store_agent.invoke(
    {"messages": [{"role": "user", "content": "Create a file named 'memo.txt' and write 'Important: the meeting is every Monday at 10 AM'."}]},
    config={**config_thread1, **lf_config},
)
print("[Thread 1] Result:")
print(result1["messages"][-1].content)
print()


In [ ]:
# Read the same note from thread 2 — confirm cross-thread access
config_thread2 = {"configurable": {"thread_id": "thread-2"}}

result2 = store_agent.invoke(
    {"messages": [{"role": "user", "content": "Is there a file named 'memo.txt'? If so, read it for me."}]},
    config={**config_thread2, **lf_config},
)
print("[Thread 2] Result:")
print(result2["messages"][-1].content)


---
## 5. `CompositeBackend` — Path-Based Routing

`CompositeBackend` routes different filesystem paths to different backends.  
A common pattern is to persist `/memories/*` while keeping everything else ephemeral.

![CompositeBackend path routing](../assets/images/composite_backend.png)


In [ ]:
composite_store, composite_checkpointer = InMemoryStore(), MemorySaver()
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("composite-demo",))},
)
composite_agent = create_deep_agent(
    model=model, backend=composite_backend, store=composite_store,
    checkpointer=composite_checkpointer, system_prompt="Use /memories/ for durable notes.",
)
print("CompositeBackend agent created")

In [ ]:
# Test persistent vs ephemeral routing
config = {"configurable": {"thread_id": "composite-test"}}

result = composite_agent.invoke(
    {"messages": [{"role": "user", "content": """
Please create two files:
1. /memories/preferences.txt — 'Preferred language: Python, editor: VS Code'
2. /scratch.txt — 'Temporary note: organize tomorrow's tasks'
Then verify that both files were created correctly.
"""}]},
    config={**config, **lf_config},
)

print(result["messages"][-1].content)


> **Note**: `CompositeBackend` removes the route prefix before storing the file internally.  
> Example: `/memories/preferences.txt` → stored internally as `/preferences.txt`  
> But the agent always accesses it with the full path (`/memories/preferences.txt`).


---
## 6. `LocalShellBackend` — Shell Execution

`LocalShellBackend` extends `FilesystemBackend` by adding shell command execution through the `execute` tool.

### ⚠️ Security Warning
> Commands run directly on the host system **with your user permissions**.  
> Use this only in development environments. In production, prefer a **sandbox backend**.

```python
from deepagents.backends import LocalShellBackend

# ⚠️ Development only!
agent = create_deep_agent(
    model=model,
    backend=LocalShellBackend(root_dir="./workspace", virtual_mode=True),
    interrupt_on={"execute": True},  # require approval before shell commands
)
```

> For safety, this notebook does **not** run `LocalShellBackend` directly.


---
## 7. Implementing a Custom Backend

If you implement `BackendProtocol`, you can build your own backend.

### Required methods

| Method | Return type | Description |
|--------|----------|------|
| `ls(path)` | `LsResult` | List directory entries |
| `read(file_path, offset=0, limit=2000)` | `ReadResult` | Read a file, or return `ReadResult(error=...)` |
| `write(file_path, content)` | `WriteResult` | Write or replace file content according to backend policy |
| `edit(file_path, old_string, new_string, replace_all=False)` | `EditResult` | Replace unique text, or report an error |
| `grep(pattern, path=None, glob=None)` | `GrepResult` | Return matches, or `GrepResult(error=...)` |
| `glob(pattern, path="/")` | `list[FileInfo]` | Return matching files |
| `delete(file_path)` | `DeleteResult` | Optional; only exposed when the backend implements deletion |

Legacy names such as `ls_info`, `grep_raw`, and `glob_info` should not be used in new teaching code.

In [ ]:
# Minimal custom backend example: read-only dictionary storage
# Uses deepagents>=0.7 BackendProtocol method and result shapes
from deepagents.backends.protocol import (
    FileInfo, LsResult, ReadResult, WriteResult, EditResult, GrepResult, GrepMatch,
)


class ReadOnlyDictBackend:
    """Read-only backend backed by an in-memory dictionary."""

    def __init__(self, files: dict[str, str]):
        self._files = files

    def ls(self, path: str = "/") -> LsResult:
        entries = [
            FileInfo(path=p, is_dir=False, size=len(c), modified_at=None)
            for p, c in self._files.items()
            if p.startswith(path)
        ]
        return LsResult(entries=sorted(entries, key=lambda e: e["path"]))

    def read(self, file_path: str, offset: int = 0, limit: int = 2000) -> ReadResult:
        content = self._files.get(file_path)
        if content is None:
            return ReadResult(error=f"File not found: {file_path}")
        lines = content.splitlines()
        selected = lines[offset:offset + limit]
        text = "\n".join(f"{i + offset + 1}\t{line}" for i, line in enumerate(selected))
        next_offset = offset + limit if offset + limit < len(lines) else None
        return ReadResult(
            file_data={"content": text, "encoding": "utf-8"},
            total_lines=len(lines), start_line=offset + 1,
            end_line=offset + len(selected), next_offset=next_offset,
            no_lines_requested=(limit == 0),
        )

    def write(self, file_path: str, content: str) -> WriteResult:
        return WriteResult(error="This backend is read-only.")

    def edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False) -> EditResult:
        return EditResult(error="This backend is read-only.")

    def grep(self, pattern: str, path: str | None = None, glob: str | None = None) -> GrepResult:
        import re
        matches = []
        for fpath, content in self._files.items():
            for i, line in enumerate(content.splitlines(), 1):
                if re.search(pattern, line):
                    matches.append(GrepMatch(path=fpath, line=i, text=line))
        return GrepResult(matches=matches)

    def glob(self, pattern: str, path: str = "/") -> list[FileInfo]:
        import fnmatch
        return [
            FileInfo(path=p, is_dir=False, size=len(c), modified_at=None)
            for p, c in self._files.items()
            if p.startswith(path) and fnmatch.fnmatch(p, pattern)
        ]


custom_backend = ReadOnlyDictBackend({
    "/docs/guide.md": "# Guide\nThis is the guide.\n## Install\npip install deepagents",
    "/docs/faq.md": "# FAQ\nQ: Which models are supported?\nA: Anthropic, OpenAI, and others.",
})

print("Files:", custom_backend.ls("/docs/"))
print()
print("Content:")
print(custom_backend.read("/docs/guide.md").file_data["content"])
print()
print("Write result:", custom_backend.write("/docs/new.md", "draft"))
print("Search result:", custom_backend.grep("Install"))

---
## Backend Selection Guide

![Backend decision guide](../assets/images/backend_decision_tree.png)


---
## Summary

| Backend | Characteristics | Parameters |
|--------|------|----------|
| `StateBackend` | Ephemeral, default | Used automatically when `backend` is omitted |
| `FilesystemBackend` | Local disk access | `root_dir`, `virtual_mode=True` by default |
| `LocalShellBackend` | Disk + shell execution | `root_dir`, `virtual_mode`, approval policy |
| `StoreBackend` | Cross-thread persistence | `namespace` + `create_deep_agent(store=...)` |
| `ContextHubBackend` | LangSmith Hub | lazy fetch + write-through cache |
| `CompositeBackend` | Path-based routing | `default` + `routes` |
| **Sandbox** | Isolated execution | Modal / Daytona / Deno / local VFS |

| `BackendProtocol` method | Return type |
|------|------|
| `ls`, `read`, `write`, `edit`, `grep`, `glob` | `LsResult`, `ReadResult`, `WriteResult`, `EditResult`, `GrepResult`, `list[FileInfo]` |
| `delete` | `DeleteResult` (optional) |

New examples pass prebuilt backend instances, and `StoreBackend` examples show both namespace and store.

## Next Step
→ **[05_subagents.ipynb](./05_subagents.ipynb)**: learn how to delegate work with subagents.